## Corpus & script

In [ ]:
!pip install openpyxl

If you want to search through *Murɣin nutenut*, comment (#...) the Weinstein subcorpus. 

```py
# CORPUS_URL = "https://raw.githubusercontent.com/katykool/ckt_hfst_annotated/refs/heads/main/charles-weinstein-morphology.jsonl"
CORPUS_URL = "https://raw.githubusercontent.com/katykool/ckt_hfst_annotated/refs/heads/main/ks-morphology.jsonl"
```

In [ ]:
SCRIPT_URL = "https://raw.githubusercontent.com/katykool/ckt_hfst_annotated/refs/heads/main/cql.py"
CORPUS_URL = "https://raw.githubusercontent.com/katykool/ckt_hfst_annotated/refs/heads/main/charles-weinstein-morphology.jsonl"

# Мургин нутэнут
# CORPUS_URL = "https://raw.githubusercontent.com/katykool/ckt_hfst_annotated/refs/heads/main/ks-morphology.jsonl"

# from google.colab import userdata
# GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
CASE_SENSITIVE = False

In [ ]:
# technical steps
import re
import sys
import urllib.request
import pandas as pd


def download(url, dest, token=None):
    req = urllib.request.Request(url)
    if token:
        req.add_header("Authorization", "token " + token)
    with urllib.request.urlopen(req) as resp, open(dest, "wb") as f:
        f.write(resp.read())
    return dest


download(SCRIPT_URL, "cql_search.py")
sys.modules.pop("cql_search", None)
import cql_search as cql

records = cql.load_corpus(download(CORPUS_URL, "corpus.jsonl"))
print("Sentences:", len(records))


Sentences: 20220


In [5]:
def search(query, limit=None, case_sensitive=CASE_SENSITIVE):
    """Returns (DataFrame, hits). Table = corpus fields + match column."""
    pattern = cql.compile_pattern(cql.parse_query(query), case_sensitive)
    hits = cql.collect_hits(records, pattern, case_sensitive)
    if limit is not None:
        hits = hits[:limit]
    header, rows = cql.build_table(hits)
    return pd.DataFrame(rows, columns=header), hits


def save(hits, path="results.xlsx"):
    """Save & download table (.xlsx или .csv)."""
    if path.lower().endswith(".xlsx"):
        cql.write_xlsx(hits, path)
    else:
        cql.write_csv(hits, path)
    print("Saved:", path)
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        pass


## 3. Query

In [6]:
QUERY = '[lemma="пыкирык|эймэвык|ӄытык|тыԓек|эквэтык|рэк|рэсӄивык|пэԓӄынтэтык|тыттэтык|рыԓык" & tag!="caus"]'

LIMIT = None      # limit the number of examples

df, hits = search(QUERY, LIMIT)
print("Совпадений:", len(df))

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
df.drop(columns=["morphology"], errors="ignore")   # morphology is hidden


Совпадений: 1294


,ckt,ru,score,source,article,translation,sent_id,match,lemma
0,"Авынвагты ыʼттъыт, аʼачекӄагтэ чейвэ аӈӄайпы ныԓейвыӄинэт","Хотя были собаки, юноши ходили на морскую охоту пешком по льду",0.659617,Charles_Weinstein,,,,ныԓейвыӄинэт,тыԓек; ԓейвык
1,"Эвынъысӄэкут, авынваӈӄаттэ ӈирэӄ ԓевмиԓеӈыт, ынӄэнат рэвытрэнӈыт","Яловые важенки, две молодые неразлучные самки, они покажутся",0.816334,Charles_Weinstein,,,,рэвытрэнӈыт,вырвыр; вытвыт; вытрэтык; рэк
2,Турыпкитԓьэт агагԓыкэгты ныԓеӄинэт,Только что приехавшие (люди) шли не торопясь,0.755337,Charles_Weinstein,,,,ныԓеӄинэт,тыԓек
3,ԓюур пыкиргъи миӈкэкин итрэтӄэй агтымкыкыԓьэн,вдруг откуда-то пришёл небольшой отряд,0.640836,Charles_Weinstein,,,,пыкиргъи,пыкирык
4,Айватԓьан-пынԓыԓьын ымы ынкы гэпкитԓин,Южанин-посыльный тоже туда пришёл,0.524260,Charles_Weinstein,,,,гэпкитԓин,пыкирык
...,...,...,...,...,...,...,...,...,...
1289,Ытԓывъёӄая пыкиринэӈу тэпӄэк йымнэнат вэԓвыегыт.,"Как только внук пришёл на косу, он надел снегоступы.",0.536804,Charles_Weinstein,,,,пыкиринэӈу,пыкирык
1290,Вуԓӄытвик ытԓён ӄытгъи яйваԓявыԓьэты.,"Вечером он пошёл к тем, которые осиротели.",0.663949,Charles_Weinstein,,,,ӄытгъи,ӄытык
1291,Мэмыԓ вэтгычемгъогъэ чьумэткук Кэӈъиринэ рээн.,Мэмыл согласился играть в шахматы с Кэӈъири.,0.712176,Charles_Weinstein,,,,рээн,риик; рук; рэк; рээн
1292,"ӄиквын ӈээкык, иӈӄун нытигыԓьэтыркын гымык рээн.","скажите дочери, пусть катается на лыжах со мной.",0.758152,Charles_Weinstein,,,,рээн,риик; рук; рэк; рээн


## 4. Results

In [7]:
save(hits, "results.xlsx")

Saved: results.xlsx
